In [ ]:
import os
import subprocess
import re
import pandas as pd

In [ ]:
os.environ["SLURM_TIME_FORMAT"] = "%s"

In [ ]:
def get_data_from_sacct(clusters: str,
                        partition: str,
                        start_date: str,
                        end_date: str,
                        fields: str) -> pd.DataFrame:
    """Return a dataframe of the sacct output."""
    cmd = f"sacct -M {clusters} -a -X -P -n -S {start_date} -E {end_date} -o {fields}"
    if partition:
        cmd += f" -r {partition}"
    output = subprocess.run(cmd,
                            stdout=subprocess.PIPE,
                            shell=True,
                            timeout=100,
                            text=True,
                            check=True)
    rows = output.stdout.strip().split('\n')
    cols = fields.split(",")
    raw = pd.DataFrame([row.split("|")[:len(cols)]
                       for row in rows if row.count("|") > len(cols) - 2])
    raw.columns = cols
    return raw

In [ ]:
def gpus_per_job(tres: str) -> int:
    """Return the number of allocated GPUs."""
    gpus = re.findall(r"gres/gpu=\d+", tres)
    return int(gpus[0].replace("gres/gpu=", "")) if gpus else 0

In [ ]:
def clean_dataframe(df: pd.DataFrame):
    """Clean the dataframe and set data types."""
    for col in ["elapsedraw", "submit", "start"]:
        df = df[pd.notna(df[col])]
        df = df[df[col].str.isnumeric()]
        df[col] = df[col].astype("int64")
    df = df[df.elapsedraw > 0]
    df["gpus"] = df.alloctres.apply(gpus_per_job)
    df["gpu-seconds"] = df.apply(lambda row:
                                 row["gpus"] * row["elapsedraw"]
                                 if row["partition"] != "mig"
                                 else row["gpus"] * row["elapsedraw"] / 7,
                                 axis="columns")
    return df

In [ ]:
clusters = "della"
partition = "gpu"
start_date = "2026-03-14T00:00:00"
end_date   = "2026-04-14T00:00:00"
fields = "user,partition,state,elapsedraw,alloctres,submit,start"

df = get_data_from_sacct(clusters, partition, start_date, end_date, fields)
df = df[(df.state != "RUNNING") | (df.state != "PENDING")]
df = clean_dataframe(df)
df = df[(df.start > 1_000_000_000) & (df.submit > 1_000_000_000)]
df = df[df.submit <= df.start]

In [ ]:
df["accrue-hrs"] = (df["start"] - df["submit"]) / 3600
df["gpu-hrs"] = df["gpu-seconds"] / 3600

In [ ]:
gp = df.groupby("user").agg({"gpu-hrs": "sum", "accrue-hrs": "sum", "user": "count"})

In [ ]:
df.info()

In [ ]:
gp.rename(columns={"user": "jobs"}, inplace=True)
gp.reset_index(inplace=True)

In [ ]:
gp.sort_values("gpu-hrs", ascending=False, inplace=True)
gp.reset_index(drop=True, inplace=True)
gp.index += 1
gp["gpu-rank"] = gp.index

In [ ]:
gp.sort_values("accrue-hrs", ascending=False, inplace=True)
gp.head(15)

In [ ]:
gp[["gpu-hrs", "accrue-hrs"]] = gp[["gpu-hrs", "accrue-hrs"]].astype("int64")
gp.reset_index(drop=True, inplace=True)
gp.index += 1

In [ ]:
gp[["user", "accrue-hrs", "gpu-hrs", "gpu-rank", "jobs"]].head(15).to_string()

In [ ]:
gp["gpu-hrs"].sum() / (31 * 356 * 24)